# LogiSense — 02 | Data Warehouse, Star Schema & ETL

### Objective
Convert the analytical source into a warehouse-oriented model suitable for OLAP.

### DWM topics
**ETL · data cleaning · fact/dimension tables · grain · surrogate keys · star schema · date dimension · referential integrity · SCD concepts · warehouse validation**

In [1]:
# ---------- LogiSense common setup ----------
from pathlib import Path
import warnings, os, json, math, re
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

def find_dataset():
    candidates = [
        Path("../data/raw/E-commerce_Delivery_Shipping_Data_2026.csv"),
        Path("../data/raw/e-commerce_delivery_shipping_data_2026.csv"),
        Path("../data/E-commerce_Delivery_Shipping_Data_2026.csv"),
        Path("../E-commerce_Delivery_Shipping_Data_2026.csv"),
        Path("E-commerce_Delivery_Shipping_Data_2026.csv"),
    ]
    for p in candidates:
        if p.exists():
            return p
    raw = Path("../data/raw")
    if raw.exists():
        csvs = list(raw.glob("*.csv"))
        if len(csvs) == 1:
            return csvs[0]
    here = Path(".")
    csvs = list(here.glob("*.csv"))
    if len(csvs) == 1:
        return csvs[0]
    raise FileNotFoundError(
        "Dataset not found. Put E-commerce_Delivery_Shipping_Data_2026.csv "
        "inside your project's data/raw/ folder."
    )

DATA_PATH = find_dataset()
df = pd.read_csv(DATA_PATH)
print(f"Loaded: {DATA_PATH}")
print(f"Shape: {df.shape}")
display(df.head(3))

Loaded: ..\data\raw\E-commerce_Delivery_Shipping_Data_2026.csv
Shape: (50000, 30)


,order_id,order_date,customer_id,customer_segment,customer_city,customer_country,warehouse_id,warehouse_city,product_category,product_weight_kg,order_value_usd,shipping_method,carrier,distance_km,promised_delivery_days,actual_delivery_days,delivery_status,late_delivery,delivery_delay_days,shipping_cost_usd,package_size,payment_method,order_priority,weather_condition,customer_rating,return_requested,return_reason,delivery_attempts,warehouse_processing_hours,tracking_status
0,ORD-000001,2026-11-15,CUS-002353,Small Business,Tokyo,Japan,WH-004,Toronto,Home & Kitchen,8.23,10.39,International,EagleCourier,1579.5,11,16,Delivered,Yes,5,133.10,Large,Digital Wallet,Normal,Snow,3,Yes,Late Delivery,2,25.1,Delivered
1,ORD-000002,2026-04-08,CUS-003258,Consumer,Vancouver,Canada,WH-007,Paris,Beauty,0.10,117.69,Standard,BlueRoute,1049.0,5,5,Delivered,No,0,26.95,Small,Debit Card,Normal,Cloudy,5,No,NaN,1,25.9,Delivered
2,ORD-000003,2026-01-05,CUS-001221,Consumer,Houston,United States,WH-005,London,Home & Kitchen,5.51,335.23,International,SpeedyCargo,4552.6,12,16,Delivered,Yes,4,338.62,Large,Debit Card,Low,Snow,4,Yes,Product Defect,1,31.5,Delivered


## 1. Define the grain

**Fact grain:** one row = one customer order/delivery event.

This is the most important warehouse design decision. Every measure in `fact_delivery` must be additive or have a clearly defined aggregation rule.

In [2]:
print("Candidate dimensions found:")
for c in ["customer_id","customer_segment","customer_city","customer_country",
          "product_category","package_size","warehouse_id","warehouse_city",
          "carrier","shipping_method","payment_method","order_priority","weather_condition"]:
    if c in df.columns:
        print("✓", c)

Candidate dimensions found:
✓ customer_id
✓ customer_segment
✓ customer_city
✓ customer_country
✓ product_category
✓ package_size
✓ warehouse_id
✓ warehouse_city
✓ carrier
✓ shipping_method
✓ payment_method
✓ order_priority
✓ weather_condition


## 2. ETL — Extract, Transform, Load

The ETL process is intentionally logged so the website can later expose data-quality status to an administrator.

In [3]:
work = df.copy()
etl_log = []

def etl_step(name, before, after, rejected=0, notes=""):
    etl_log.append({
        "step": name, "rows_before": before, "rows_after": after,
        "rejected": rejected, "notes": notes
    })

etl_step("Extract", len(work), len(work), notes="Raw CSV loaded")

# Standardize date
if "order_date" in work.columns:
    before = len(work)
    work["order_date"] = pd.to_datetime(work["order_date"], errors="coerce")
    rejected = work["order_date"].isna().sum()
    work = work.dropna(subset=["order_date"]).copy()
    etl_step("Date validation", before, len(work), int(rejected), "Invalid order dates removed")

# Numeric non-negative checks
numeric_operational = [c for c in [
    "product_weight_kg","order_value_usd","distance_km","promised_delivery_days",
    "actual_delivery_days","delivery_delay_days","shipping_cost_usd",
    "customer_rating","delivery_attempts","warehouse_processing_hours"
] if c in work.columns]

before = len(work)
bad = pd.Series(False, index=work.index)
for c in numeric_operational:
    bad |= pd.to_numeric(work[c], errors="coerce").lt(0)
work = work.loc[~bad].copy()
etl_step("Range validation", before, len(work), int(bad.sum()),
         "Removed negative operational measures")

# Duplicate business key
if "order_id" in work.columns:
    before = len(work)
    dup = work["order_id"].duplicated(keep="first")
    work = work.loc[~dup].copy()
    etl_step("Deduplication", before, len(work), int(dup.sum()),
             "Kept first row per order_id")

display(pd.DataFrame(etl_log))

,step,rows_before,rows_after,rejected,notes
0,Extract,50000,50000,0,Raw CSV loaded
1,Date validation,50000,50000,0,Invalid order dates removed
2,Range validation,50000,50000,0,Removed negative operational measures
3,Deduplication,50000,50000,0,Kept first row per order_id


## 3. Date Dimension

The date dimension enables warehouse OLAP hierarchies:
**Day → Month → Quarter → Year**.

In [4]:
if "order_date" in work.columns:
    dim_date = work[["order_date"]].drop_duplicates().sort_values("order_date").reset_index(drop=True)
    dim_date.insert(0,"date_key",range(1,len(dim_date)+1))
    dim_date["day"] = dim_date.order_date.dt.day
    dim_date["day_name"] = dim_date.order_date.dt.day_name()
    dim_date["week"] = dim_date.order_date.dt.isocalendar().week.astype(int)
    dim_date["month"] = dim_date.order_date.dt.month
    dim_date["month_name"] = dim_date.order_date.dt.month_name()
    dim_date["quarter"] = "Q" + dim_date.order_date.dt.quarter.astype(str)
    dim_date["year"] = dim_date.order_date.dt.year
    dim_date["is_weekend"] = dim_date.order_date.dt.dayofweek >= 5
    display(dim_date.head())

,date_key,order_date,day,day_name,week,month,month_name,quarter,year,is_weekend
0,1,2026-01-01,1,Thursday,1,1,January,Q1,2026,False
1,2,2026-01-02,2,Friday,1,1,January,Q1,2026,False
2,3,2026-01-03,3,Saturday,1,1,January,Q1,2026,True
3,4,2026-01-04,4,Sunday,1,1,January,Q1,2026,True
4,5,2026-01-05,5,Monday,2,1,January,Q1,2026,False


## 4. Dimension builders

Surrogate integer keys are used so the fact table is independent of source-system identifiers.

In [5]:
def make_dim(cols, key):
    cols=[c for c in cols if c in work.columns]
    d=work[cols].drop_duplicates().reset_index(drop=True)
    d.insert(0,key,range(1,len(d)+1))
    return d

dim_customer = make_dim(["customer_id","customer_segment","customer_city","customer_country"],"customer_key")
dim_product = make_dim(["product_category","package_size"],"product_key")
dim_warehouse = make_dim(["warehouse_id","warehouse_city"],"warehouse_key")
dim_carrier = make_dim(["carrier"],"carrier_key")
dim_shipping = make_dim(["shipping_method","payment_method","order_priority"],"shipping_key")
dim_weather = make_dim(["weather_condition"],"weather_key")

for name,t in [
    ("dim_customer",dim_customer),("dim_product",dim_product),
    ("dim_warehouse",dim_warehouse),("dim_carrier",dim_carrier),
    ("dim_shipping",dim_shipping),("dim_weather",dim_weather)
]:
    print(name, t.shape)

dim_customer (48900, 5)
dim_product (45, 3)
dim_warehouse (10, 3)
dim_carrier (8, 2)
dim_shipping (120, 4)
dim_weather (6, 2)


## 5. Fact table construction

Only numeric/flag measures and foreign keys belong in the central fact table.

In [6]:
fact = work.copy()

if "order_date" in fact.columns:
    fact = fact.merge(dim_date[["date_key","order_date"]], on="order_date", how="left")

merge_specs = [
    (dim_customer, ["customer_id","customer_segment","customer_city","customer_country"]),
    (dim_product, ["product_category","package_size"]),
    (dim_warehouse, ["warehouse_id","warehouse_city"]),
    (dim_carrier, ["carrier"]),
    (dim_shipping, ["shipping_method","payment_method","order_priority"]),
    (dim_weather, ["weather_condition"])
]

for dim, keys in merge_specs:
    keys=[k for k in keys if k in fact.columns and k in dim.columns]
    if keys:
        fact=fact.merge(dim,on=keys,how="left")

fact_cols = [c for c in [
    "order_id","date_key","customer_key","product_key","warehouse_key","carrier_key",
    "shipping_key","weather_key","product_weight_kg","order_value_usd","distance_km",
    "promised_delivery_days","actual_delivery_days","delivery_delay_days",
    "shipping_cost_usd","customer_rating","delivery_attempts",
    "warehouse_processing_hours"
] if c in fact.columns]

fact_delivery=fact[fact_cols].copy()
fact_delivery.insert(0,"delivery_key",range(1,len(fact_delivery)+1))

if "late_delivery" in work.columns:
    fact_delivery["late_flag"]=work.loc[fact_delivery.index,"late_delivery"].astype(str).str.lower().eq("yes").astype(int).values
if "return_requested" in work.columns:
    fact_delivery["return_flag"]=work.loc[fact_delivery.index,"return_requested"].astype(str).str.lower().eq("yes").astype(int).values

display(fact_delivery.head())
print("Fact grain rows:", len(fact_delivery))

,delivery_key,order_id,date_key,customer_key,product_key,warehouse_key,carrier_key,shipping_key,weather_key,product_weight_kg,order_value_usd,distance_km,promised_delivery_days,actual_delivery_days,delivery_delay_days,shipping_cost_usd,customer_rating,delivery_attempts,warehouse_processing_hours,late_flag,return_flag
0,1,ORD-000001,319,1,1,1,1,1,1,8.23,10.39,1579.5,11,16,5,133.10,3,2,25.1,1,1
1,2,ORD-000002,98,2,2,2,2,2,2,0.10,117.69,1049.0,5,5,0,26.95,5,1,25.9,0,0
2,3,ORD-000003,5,3,1,3,3,3,1,5.51,335.23,4552.6,12,16,4,338.62,4,1,31.5,1,1
3,4,ORD-000004,149,4,3,4,4,2,1,3.75,10.57,7074.9,8,11,3,270.45,3,1,12.6,1,0
4,5,ORD-000005,53,5,2,5,5,4,3,0.10,31.37,522.0,8,12,4,12.90,2,1,24.9,1,0


Fact grain rows: 50000


## 6. Referential-integrity validation

A warehouse should never silently contain orphan foreign keys.

In [7]:
fk_checks=[]
for fk,dim in [
    ("date_key",dim_date),("customer_key",dim_customer),("product_key",dim_product),
    ("warehouse_key",dim_warehouse),("carrier_key",dim_carrier),
    ("shipping_key",dim_shipping),("weather_key",dim_weather)
]:
    if fk in fact_delivery.columns:
        missing=fact_delivery[fk].isna().sum()
        fk_checks.append([fk,len(dim),int(missing),100*missing/len(fact_delivery)])
display(pd.DataFrame(fk_checks,columns=["foreign_key","dimension_rows","missing_fk","missing_%"]).round(3))

,foreign_key,dimension_rows,missing_fk,missing_%
0,date_key,365,0,0.0
1,customer_key,48900,0,0.0
2,product_key,45,0,0.0
3,warehouse_key,10,0,0.0
4,carrier_key,8,0,0.0
5,shipping_key,120,0,0.0
6,weather_key,6,0,0.0


## 7. Star-schema relationship map

```text
                 dim_date
                    |
dim_customer — fact_delivery — dim_product
                    |
              dim_warehouse
                    |
               dim_carrier
                    |
              dim_shipping
                    |
              dim_weather
```

This is intentionally a **star schema**, not a highly normalized transactional model. It is optimized for analytical aggregation.

## 8. SCD strategy

For a production implementation:

- **Type 1:** use when only the latest attribute matters.
- **Type 2:** preferred when historical customer segment, warehouse assignment or business classification must remain historically accurate.
- **Type 3:** useful for keeping a limited previous value.

For this mini-project, the source is a historical snapshot, so a Type-2 framework is documented rather than artificially inventing history.

## 9. Export warehouse-ready tables

These CSVs can be loaded into PostgreSQL/MySQL/DuckDB for the website backend.

In [8]:
OUT=Path("../data/processed")
OUT.mkdir(parents=True,exist_ok=True)

tables={
    "dim_date":dim_date,
    "dim_customer":dim_customer,
    "dim_product":dim_product,
    "dim_warehouse":dim_warehouse,
    "dim_carrier":dim_carrier,
    "dim_shipping":dim_shipping,
    "dim_weather":dim_weather,
    "fact_delivery":fact_delivery
}
for name,t in tables.items():
    p=OUT/f"{name}.csv"
    t.to_csv(p,index=False)
    print("Saved",p,t.shape)

Saved ..\data\processed\dim_date.csv (365, 10)
Saved ..\data\processed\dim_customer.csv (48900, 5)
Saved ..\data\processed\dim_product.csv (45, 3)
Saved ..\data\processed\dim_warehouse.csv (10, 3)
Saved ..\data\processed\dim_carrier.csv (8, 2)
Saved ..\data\processed\dim_shipping.csv (120, 4)
Saved ..\data\processed\dim_weather.csv (6, 2)
Saved ..\data\processed\fact_delivery.csv (50000, 21)
